# RAG System Core Experiments

This notebook demonstrates the core RAG (Retrieval-Augmented Generation) pipeline extracted from the fullstack chatbot system.

## Pipeline Overview

1. **Document Loading**: Load markdown documents
2. **Chunking**: Split documents into semantic chunks with overlap
3. **Embedding**: Generate embeddings using OpenAI
4. **Vector Indexing**: Store embeddings in ChromaDB
5. **Retrieval**: Search for relevant chunks
6. **Generation**: Generate answers using LLM

## Setup

In [1]:
import os
import sys

# Add current directory to path
sys.path.insert(0, os.getcwd())

# Import core modules
from config import config
from document_loader import DocumentLoader
from chunker import SemanticChunker
from embedding_service import EmbeddingService
from rag_pipeline import RAGPipeline

print("Core modules imported successfully!")
print(f"OpenAI API Key configured: {bool(config.OPENAI_API_KEY)}")

Core modules imported successfully!
OpenAI API Key configured: True


## Step 1: Document Loading

Load markdown documents from the file system.

In [ ]:
# Initialize document loader
loader = DocumentLoader()

# Example: Load a single document
# Replace with your document path
document_paths = [
    # Add your markdown file paths here
    # Example: '/path/to/your/document.md'
]

# If you don't have documents yet, create a sample
if not document_paths:
    print("No documents specified. Creating sample document...")
    
    sample_doc_path = './sample_document.md'
    sample_content = '''---
document_type: tutorial
domain: actuarial
---

# Introduction to Actuarial Science

Actuarial science is the discipline that applies mathematical and statistical methods to assess risk in insurance, finance, and other industries.

## Key Concepts

### Probability Theory
Probability theory is fundamental to actuarial calculations. It helps in predicting future events and quantifying uncertainty.

### Risk Assessment
Risk assessment involves identifying, analyzing, and evaluating potential risks. Actuaries use various models to assess risks accurately.

### Financial Mathematics
Financial mathematics deals with time value of money, interest rates, and investment returns. These concepts are crucial for pension planning and life insurance.

## Applications

Actuarial science is applied in:
- Life insurance
- Health insurance
- Pension funds
- Investment management
- Risk management
'''
    with open(sample_doc_path, 'w', encoding='utf-8') as f:
        f.write(sample_content)
    
    document_paths = [sample_doc_path]
    print(f"Sample document created at: {sample_doc_path}")

# Load documents
documents = loader.load_documents(document_paths)

print(f"\nLoaded {len(documents)} document(s):")
for doc in documents:
    print(f"  - {doc.filename}: {len(doc.content)} characters")
    print(f"    Metadata: {doc.metadata}")

nggih niat sampeyan apik, cuman aku isok lebih tekan iku

## Step 2: Document Chunking

Split documents into semantic chunks with overlap for better retrieval.

In [ ]:
# Initialize chunker
chunker = SemanticChunker(
    max_chunk_size=1000,
    overlap_size=150
)

# Chunk all documents
all_chunks = []
for doc in documents:
    chunks = chunker.chunk_document(
        content=doc.content,
        metadata=doc.metadata,
        doc_name=doc.filename
    )
    all_chunks.extend(chunks)
    
    print(f"\n{doc.filename}:")
    print(f"  Created {len(chunks)} chunks")
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n  Chunk {i+1} ({chunk.token_count} tokens):")
        preview = chunk.content[:150].replace('\n', ' ')
        print(f"    {preview}...")

print(f"\n\nTotal chunks created: {len(all_chunks)}")

## Step 3: Embedding and Vector Indexing

Generate embeddings and store in ChromaDB vector database.

In [ ]:
# Initialize embedding service
embedding_service = EmbeddingService(
    embedding_model=config.EMBEDDING_MODEL,
    chroma_path=config.CHROMA_DB_PATH,
    collection_name=config.COLLECTION_NAME
)

# Optional: Clear existing collection
# embedding_service.clear_collection()

# Add chunks to vector store
embedding_service.add_chunks(all_chunks)

# Get collection stats
stats = embedding_service.get_collection_stats()
print("\nVector Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## Step 4: Similarity Search

Test retrieval by searching for relevant chunks.

In [ ]:
# Test query
test_query = "What is probability theory in actuarial science?"

# Perform similarity search
results = embedding_service.similarity_search(
    query=test_query,
    k=3
)

print(f"Query: {test_query}")
print(f"\nFound {len(results)} relevant chunks:\n")

for i, (doc, score) in enumerate(results):
    print(f"Result {i+1} (Score: {score:.4f}):")
    print(f"  Source: {doc.metadata.get('filename', 'Unknown')}")
    print(f"  Chunk ID: {doc.metadata.get('chunk_id', 'Unknown')}")
    preview = doc.page_content[:200].replace('\n', ' ')
    print(f"  Content: {preview}...")
    print()

## Step 5: RAG Pipeline - Question Answering

Use the complete RAG pipeline to answer questions.

In [ ]:
# Initialize RAG pipeline
rag = RAGPipeline(
    embedding_service=embedding_service,
    llm_model=config.OPENAI_MODEL,
    temperature=config.TEMPERATURE,
    max_context_length=config.MAX_CONTEXT_LENGTH
)

print("RAG Pipeline initialized successfully!")

In [ ]:
# Ask questions
questions = [
    "What is actuarial science?",
    "Explain probability theory in actuarial context",
    "What are the applications of actuarial science?"
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    # Get answer from RAG pipeline
    response = rag.query(
        question=question,
        k=5,
        return_sources=True
    )
    
    print(f"\nAnswer:\n{response['answer']}")
    print(f"\nConfidence: {response['confidence']:.2f}")
    
    if 'sources' in response and response['sources']:
        print(f"\nSources ({len(response['sources'])}):")  
        for i, source in enumerate(response['sources'][:3]):
            print(f"  {i+1}. {source['metadata'].get('filename', 'Unknown')} (Score: {source['score']:.3f})")

## Experiment: Different Chunking Sizes

Compare performance with different chunk sizes.

In [ ]:
# Experiment with different chunk sizes
chunk_sizes = [500, 1000, 1500]
test_query = "What is risk assessment?"

for size in chunk_sizes:
    print(f"\n{'='*60}")
    print(f"Testing with chunk size: {size} tokens")
    print(f"{'='*60}")
    
    # Create new chunker
    test_chunker = SemanticChunker(max_chunk_size=size, overlap_size=150)
    
    # Chunk documents
    test_chunks = []
    for doc in documents:
        chunks = test_chunker.chunk_document(doc.content, doc.metadata, doc.filename)
        test_chunks.extend(chunks)
    
    print(f"  Total chunks: {len(test_chunks)}")
    avg_size = sum(c.token_count for c in test_chunks) / len(test_chunks) if test_chunks else 0
    print(f"  Avg chunk size: {avg_size:.1f} tokens")

## Summary

This notebook demonstrated the core RAG pipeline:

1. Document loading with YAML front-matter support
2. Semantic chunking with overlap
3. Embedding generation (OpenAI)
4. Vector indexing (ChromaDB)
5. Similarity search
6. RAG-based question answering

### Next Steps

- Load your own documents
- Experiment with different chunking strategies
- Try different embedding models
- Implement BM25 hybrid search
- Add evaluation metrics